# InsPLAD Final Gold QA — Vision Expert Smoke

현재 고정된 `gold_v1`에서 Vision Expert가 실제로 LVLM에 도움이 되는지 방법 선택용 smoke를 수행합니다.

- 250장 중 asset별 10장씩, 총 50장을 deterministic dev cohort로 고정
- Vision Expert: PatchCore / ReConPatch-lite
- Evidence: ROI crop / heatmap overlay
- LVLM: Qwen3.5-4B / LLaVA-OneVision-7B / MiniCPM-V-4.5
- Q1/Q2/Q3는 기존 gold QA 문항 그대로 사용

이 50장은 방법 선택용이며 최종 논문 성능은 아닙니다. winner를 고른 뒤 전체 benchmark에 확장합니다.


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v1"
RESULT_FOLDER = "lvlm_insplad_results/gold_vision_expert_smoke_v1"
DRIVE_DATA_DIR = None

MODELS_TO_RUN = ["qwen3_5_4b", "llava_onevision_7b", "minicpm_v_4_5"]
PER_ASSET = 10
SEED = 42
MAX_TRAIN_NORMALS_PER_ASSET = 200
PATCHCORE_CORESET_RATIO = 0.05
RECONPATCH_TRAIN_STEPS = 120
RECONPATCH_MAX_TRAIN_PATCHES = 6000
RECONPATCH_MEMORY_RATIO = 0.05
CHECKPOINT_EVERY = 5

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
SUP_ZIP_NAME = "supervised_fault_classification.zip"
EXPECTED_ZIP_SIZES = {UNSUP_ZIP_NAME: 1_170_967_719, SUP_ZIP_NAME: 874_753_167}


In [ ]:
# 1) Drive mount + private repo clone + dependencies
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
QA_DIR = DRIVE_ROOT / QA_FOLDER
RESULT_DIR = DRIVE_ROOT / RESULT_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/lvlm_insplad')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None):
    cmd=[str(x) for x in cmd]; print('+', ' '.join(cmd[:8]) + (' ...' if len(cmd)>8 else ''))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

try: github_token = userdata.get('GITHUB_TOKEN')
except Exception: github_token = None
if not github_token: raise RuntimeError('Colab Secret GITHUB_TOKEN이 없습니다.')
auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env=os.environ.copy(); git_env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}','GIT_TERMINAL_PROMPT':'0'})
repo_url=f'https://github.com/{REPO}.git'
probe=subprocess.run(['git','ls-remote','--heads',repo_url,REPO_REF],env=git_env,capture_output=True,text=True)
if probe.returncode != 0: raise RuntimeError(f"GitHub 인증 실패: {(probe.stderr or '').strip()[:1200]}")
run(['git','clone','--depth','1','--branch',REPO_REF,repo_url,REPO_DIR], env=git_env)
del auth, git_env, github_token

run([sys.executable,'-m','pip','install','-q',
     'transformers==5.14.0','accelerate==1.14.0','sentencepiece==0.2.2',
     'anomalib==2.6.0','opencv-python-headless>=4.10','scipy>=1.12','ftfy>=6.2','regex>=2024.0',
     'lightning','timm','pandas','pillow','tqdm'])
print('repo:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip())


In [ ]:
# 2) Secrets + GPU
import torch
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('PASS HF_TOKEN')
else:
    print('WARN HF_TOKEN 없음')
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime이 필요합니다.')
props=torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GB:', round(props.total_memory/1024**3,1))


In [ ]:
# 3) source ZIP 찾기 + manifest 준비
import pandas as pd

def find_unique_file(filename):
    roots=[DRIVE_ROOT / DRIVE_DATA_DIR] if DRIVE_DATA_DIR else []
    roots.append(DRIVE_ROOT); matches=[]
    for root in roots:
        if not root.exists(): continue
        matches += [p for p in root.rglob(filename) if p.is_file()]
    uniq=[]; seen=set()
    for p in matches:
        k=str(p.resolve())
        if k not in seen: uniq.append(p); seen.add(k)
    if not uniq: raise FileNotFoundError(filename)
    if len(uniq)>1: print('[주의] 여러 후보 중 첫 파일:', uniq[0])
    return uniq[0]

UNSUP_ZIP=find_unique_file(UNSUP_ZIP_NAME); SUP_ZIP=find_unique_file(SUP_ZIP_NAME)
for p in [UNSUP_ZIP,SUP_ZIP]:
    if p.stat().st_size != EXPECTED_ZIP_SIZES[p.name]: raise RuntimeError(f'ZIP size mismatch: {p}')
    print('PASS ZIP:', p)

MANIFEST_DIR=RESULT_DIR/'manifests'; MANIFEST_DIR.mkdir(exist_ok=True)
UNSUP_MANIFEST=MANIFEST_DIR/'unsupervised_manifest.csv'
if not UNSUP_MANIFEST.is_file():
    run([sys.executable,REPO_DIR/'scripts/inspect_zip_manifest.py',UNSUP_ZIP,'--out-csv',UNSUP_MANIFEST,'--out-json',MANIFEST_DIR/'unsupervised_manifest_summary.json'])
print('manifest:', UNSUP_MANIFEST)


In [ ]:
# 4) gold_v1에서 방법 선택용 50장 고정
SELECT_DIR=RESULT_DIR/'selection'
run([sys.executable, REPO_DIR/'scripts/build_gold_vision_expert_smoke.py',
     '--qa-dir',QA_DIR,'--out-dir',SELECT_DIR,'--per-asset',str(PER_ASSET),'--seed',str(SEED)])
SMOKE_IMAGES=SELECT_DIR/'expert_smoke_images.jsonl'
SMOKE_MCQA=SELECT_DIR/'expert_smoke_mcqa.jsonl'
display(pd.read_csv(SELECT_DIR/'selection_summary.csv'))


In [ ]:
# 5) leakage-safe 정상 train pool + 50 target workspace
WORKSPACE=RESULT_DIR/'workspace'
run([sys.executable, REPO_DIR/'scripts/build_vision_expert_smoke_data.py',
     '--anomaly-manifest',UNSUP_MANIFEST,'--unsup-zip',UNSUP_ZIP,'--qa',SMOKE_IMAGES,
     '--out-dir',WORKSPACE,'--max-train-per-asset',str(MAX_TRAIN_NORMALS_PER_ASSET),'--seed',str(SEED)])
print('workspace ready:', WORKSPACE)


In [ ]:
# 6) Vision Expert A — PatchCore
PATCH_ROOT=RESULT_DIR/'patchcore'
run([sys.executable, REPO_DIR/'scripts/run_patchcore_cohort.py',
     '--workspace-root',WORKSPACE,'--out-dir',PATCH_ROOT,
     '--coreset-ratio',str(PATCHCORE_CORESET_RATIO),'--seed',str(SEED)])
PATCH_MANIFEST=PATCH_ROOT/'roi_manifest.csv'
print('PatchCore:', PATCH_MANIFEST)


In [ ]:
# 7) Vision Expert B — ReConPatch-lite
RECON_ROOT=RESULT_DIR/'reconpatch_lite'
run([sys.executable, REPO_DIR/'scripts/run_reconpatch_lite_smoke.py',
     '--workspace-root',WORKSPACE,'--out-dir',RECON_ROOT,
     '--train-steps',str(RECONPATCH_TRAIN_STEPS),'--max-train-patches',str(RECONPATCH_MAX_TRAIN_PATCHES),
     '--memory-ratio',str(RECONPATCH_MEMORY_RATIO),'--seed',str(SEED)])
RECON_MANIFEST=RECON_ROOT/'roi_manifest_reconpatch_lite.csv'
print('ReConPatch-lite:', RECON_MANIFEST)


In [ ]:
# 8) 동일 MCQA에서 ROI vs Heatmap 비교
EVAL_DIR=RESULT_DIR/'mcqa'
run([sys.executable, REPO_DIR/'scripts/run_gold_vision_expert_mcqa.py',
     '--qa',SMOKE_MCQA,'--sup-zip',SUP_ZIP,'--out-dir',EVAL_DIR,
     '--patchcore-manifest',PATCH_MANIFEST,'--patchcore-root',PATCH_ROOT,
     '--reconpatch-manifest',RECON_MANIFEST,'--reconpatch-root',RECON_ROOT,
     '--models',*MODELS_TO_RUN,'--conditions','roi','heatmap','--checkpoint-every',str(CHECKPOINT_EVERY)])


In [ ]:
# 9) 결과 확인
lb=pd.read_csv(EVAL_DIR/'leaderboard.csv')
summary=pd.read_csv(EVAL_DIR/'screening_summary.csv')
print('방법 선택 요약')
display(summary)
print('모델별 상세')
display(lb.sort_values(['anomaly_f1','defect_macro_f1'],ascending=False))
print('run status')
print((EVAL_DIR/'run_status.json').read_text(encoding='utf-8'))


In [ ]:
# 10) 핵심 결과를 MyDrive 루트에도 publish — ChatGPT가 바로 찾기 쉽게
sys.path.insert(0, str(REPO_DIR/'scripts'))
from publish_latest_results import publish
published=publish(EVAL_DIR,'lvlm_gold_vision_expert',(
    'leaderboard.csv','screening_summary.csv','metrics_by_asset.csv','run_status.json','run_metadata.json'
))
print(json.dumps(published,ensure_ascii=False,indent=2))


## 다음 판단

- `screening_summary.csv`의 `mean_primary_score`(Anomaly F1 + Defect Macro-F1 평균)가 가장 높은 expert/evidence 조합을 1차 winner로 선택합니다.
- Localization은 현재 Q3 subset 범위가 제한적이므로 별도로 확인합니다.
- winner가 정해지면 그 조합만 전체 gold benchmark로 확장합니다.
- 그 다음에 `winner + Normal Reference`를 붙여 Reference가 Vision Expert의 약점을 보완하는지 확인합니다.
- ReConPatch-lite가 이기면 현재 구현은 screening용이므로 논문 최종 주장 전에 publication-grade 구현으로 재검증합니다.
